In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 17 samples and 13911 columns.
First few rows of the data:


,months to death or last fu,status (alive - 0; dead/hospice - 1),sample_title,tumor type,"timing of immunotherapy use relative to rt (1 - after only, 2 - before and after)",OR4F29,OR4F16,NOC2L,KLHL17,PLEKHN1,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,22.566667,0,"SM-HCP (YY, YZ, Z1)",Metastatic Melanoma,1,1.419764,1.791127,3.737855,1.903591,2.064060,...,2.019139,0.822458,2.145742,1.870827,1.116687,0.611674,2.993061,2.801203,1.652033,2.512309
1,2.800000,1,SM-HCP (ZA-ZD),Metastatic Melanoma,2,1.018639,1.466905,3.547749,2.360976,0.528370,...,2.406441,1.182506,2.290120,2.175077,1.330957,1.099105,3.469617,3.294399,2.272340,2.570081
2,24.233333,1,SM-HCQ6 (S-V),Metastatic Melanoma,1,1.729009,1.896278,3.376944,1.931879,1.375554,...,1.673971,0.317861,1.191918,1.429004,0.612444,0.245000,2.238251,1.709542,0.729611,1.603659
3,16.033333,1,SM-HCP (WL-WP),Metastatic Melanoma,2,0.640884,1.105541,4.492214,3.184389,1.651799,...,3.288294,1.306839,2.585870,2.523845,1.427557,1.513530,3.765575,3.539695,2.527836,2.943814
4,51.966667,0,SM-HCQ7 (D-L),Metastatic Melanoma,1,0.782545,1.759684,4.663638,2.816783,0.801410,...,2.272631,1.077299,2.308460,2.089459,0.459387,1.406616,2.741814,2.622187,1.621544,2.399429


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for months to death or last fu or status (alive - 0; dead/hospice - 1): 0
Number of samples with complete data for both variables: 17
Number of censored samples (event_var == 0.0): 9
Number of events (event_var == 1.0): 8
Censored ratio (among complete): 9 / 17 = 0.529 (52.9%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE131521/description.json
